In [1]:
!pip install matplotlib scipy

In [2]:
import sys
!{sys.executable} -m pip install scipy

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

# Set random seed for reproducibility
np.random.seed(42)

# Suppose we have n = 3 random variables, with N = 100000 total samples
# X_data shape is (3, 100000), where each row represents samples of one random variable
X_data = np.random.randn(3, 100000)

# ----------------------------------------------------
# 1. Verify Expectation (Mean Vector)
# ----------------------------------------------------
# Manual calculation: compute the mean along each row (axis=1)
E_X_manual = np.mean(X_data, axis=1, keepdims=True)

print("--- Expectation Vector E[X] ---")
print(E_X_manual.flatten())

--- Expectation Vector E[X] ---
[ 0.00096687  0.00098103 -0.00150833]


In [7]:
# ----------------------------------------------------
# 2. Verify Covariance Matrix
# ----------------------------------------------------
# Formula: Sigma = E[(X - E[X])(X - E[X])^T]
# Step 1: Center the matrix (X - E[X])
X_centered = X_data - E_X_manual

# Step 2: Matrix multiplication and take the expectation 
# (Divided by N to match the theoretical formula)
# X_centered shape is (3, N), its transpose is (N, 3), resulting in (3, 3)
Sigma_manual = (X_centered @ X_centered.T) / X_data.shape[1]

# Use NumPy's built-in np.cov 
# (Default is unbiased estimation ddof=1, set ddof=0 to match the formula)
Sigma_numpy = np.cov(X_data, ddof=0)

print("\n--- Manually Calculated Covariance Matrix ---")
print(Sigma_manual)

print("\n--- NumPy Built-in Covariance Matrix ---")
print(Sigma_numpy)


--- Manually Calculated Covariance Matrix ---
[[ 1.00180272  0.00632267  0.00196039]
 [ 0.00632267  0.99786979 -0.00381868]
 [ 0.00196039 -0.00381868  0.99969458]]

--- NumPy Built-in Covariance Matrix ---
[[ 1.00180272  0.00632267  0.00196039]
 [ 0.00632267  0.99786979 -0.00381868]
 [ 0.00196039 -0.00381868  0.99969458]]


In [11]:
# =============================================================================
# 1. Random Vectors, Expectation, and Covariance Matrix Verification
# =============================================================================
print("=== 1. Random Vectors and Covariance Matrix Verification ===")
n_dims = 3  # Define the dimensionality of the random vector (n = 3)
true_mean = np.array([2.0, -1.0, 0.5])

# Construct a valid symmetric positive definite matrix for the true covariance
A = np.random.randn(n_dims, n_dims)
true_cov = np.dot(A, A.T) + 0.5 * np.eye(n_dims)

# Simulate generating N samples of the random vector, shape: (N_samples, n_dims)
N_samples = 100000
X = np.random.multivariate_normal(true_mean, true_cov, size=N_samples)

# (1) Calculate Sample Expectation E[X]
sample_mean = np.mean(X, axis=0)
print(f"Theoretical Mean E[X]: {true_mean}")
print(f"Sample Mean:            {sample_mean}\n")

# (2) Compute Covariance Matrix from scratch: \Sigma = E[(X-E[X])(X-E[X])^T]
X_centered = X - sample_mean
sample_cov_formula = (X_centered.T @ X_centered) / (N_samples - 1)
np_cov_result = np.cov(X,rowvar=False)
print("Covariance Matrix Σ calculated via formula:\n", sample_cov_formula)
print("\nNumPy built-in np.cov result:\n", np_cov_result)
print("\nTheoretical Covariance:\n", true_cov)

# (3) Verify Proposition 4.2: Covariance matrix is symmetric positive semidefinite
is_symmetric = np.allclose(sample_cov_formula, sample_cov_formula.T)
eigenvalues = np.linalg.eigvals(sample_cov_formula)
is_psd = np.all(eigenvalues >= -1e-10)  # Tolerate minor floating-point errors

print(f"\n[Verification] Is the covariance matrix symmetric? {is_symmetric}")
print(f"[Verification] Eigenvalues of the covariance matrix: {eigenvalues}")
print(f"[Verification] Are all eigenvalues >= 0 (Positive Semidefinite)? {is_psd}\n")

=== 1. Random Vectors and Covariance Matrix Verification ===
Theoretical Mean E[X]: [ 2.  -1.   0.5]
Sample Mean:            [ 2.00445974 -0.99244612  0.49515582]

Covariance Matrix Σ calculated via formula:
 [[ 3.62531186 -2.69907086 -0.40211379]
 [-2.69907086  4.56187591  0.05505607]
 [-0.40211379  0.05505607  0.82661296]]

NumPy built-in np.cov result:
 [[ 3.62531186 -2.69907086 -0.40211379]
 [-2.69907086  4.56187591  0.05505607]
 [-0.40211379  0.05505607  0.82661296]]

Theoretical Covariance:
 [[ 3.62571392 -2.69184689 -0.40603509]
 [-2.69184689  4.51929913  0.05597543]
 [-0.40603509  0.05597543  0.8313159 ]]

[Verification] Is the covariance matrix symmetric? True
[Verification] Eigenvalues of the covariance matrix: [6.84806891 1.46696379 0.69876803]
[Verification] Are all eigenvalues >= 0 (Positive Semidefinite)? True



In [17]:
# =============================================================================
# 2. Law of Large Numbers (LLN) Verification
# =============================================================================
print("=== 2. Law of Large Numbers Verification ===")
lam = 0.5
theoretical_mean = 1.0 / lam  # Theoretical mean of Exponential distribution is 2.0

N_lln = 5000
# Generate i.i.d. samples from an Exponential distribution
samples_lln = np.random.exponential(scale=1/lam, size=N_lln)

# Compute cumulative running average: [X1, (X1+X2)/2, (X1+X2+X3)/3, ...]
running_average = np.cumsum(samples_lln) / np.arange(1, N_lln + 1)

# Plot LLN convergence
plt.figure(figsize=(10, 5))
plt.plot(running_average, label='Sample Average (Running Mean)', color='blue', lw=1.5)
plt.axhline(y=theoretical_mean, color='red', linestyle='--', label=rf'Theoretical Mean ($\mu$ = {theoretical_mean})')
plt.title('Law of Large Numbers (LLN) Verification', fontsize=12)
plt.xlabel('Number of Samples ($n$)')
plt.ylabel('Value')
plt.legend()
plt.grid(True, linestyle=':')
plt.savefig('lln_verification.png', bbox_inches='tight')
plt.close()
print("LLN visualization saved as 'lln_verification.png'\n")


=== 2. Law of Large Numbers Verification ===
LLN visualization saved as 'lln_verification.png'



In [4]:
# =============================================================================
# 3. Central Limit Theorem (CLT) Verification
# =============================================================================
print("=== 3. Central Limit Theorem Verification ===")
# Use Uniform(0, 1) as the underlying base distribution
mu_clt = 0.5
sigma_clt = np.sqrt(1/12)

sample_sizes = [2, 5, 30]  # Observe the impact of sample size n
n_simulations = 10000      # Number of independent simulation trials

plt.figure(figsize=(15, 4))

for idx, n in enumerate(sample_sizes):
    # Generate a matrix of shape (n_simulations, n)
    # Each row represents a single trial containing n samples
    trials = np.random.uniform(0, 1, size=(n_simulations, n))
    
    # Compute the sample mean for each trial
    sample_means = np.mean(trials, axis=1)
    
    # Normalize according to formula (14) to obtain \xi_n
    normalized_means = (sample_means - mu_clt) / (sigma_clt / np.sqrt(n))
    
    # Plot the probability density histogram of the simulated \xi_n
    plt.subplot(1, 3, idx + 1)
    plt.hist(normalized_means, bins=50, density=True, alpha=0.6, color='darkcyan', edgecolor='black')
    
    # Plot the theoretical Standard Normal PDF for comparison
    x_axis = np.linspace(-4, 4, 100)
    plt.plot(x_axis, stats.norm.pdf(x_axis, 0, 1), 'r-', lw=2, label=r'Standard Normal $\Phi(x)$')
    
    plt.title(f'Sample Size $n = {n}$')
    plt.xlabel(r'$\xi_n$')
    plt.ylabel('Density')
    if idx == 0:
        plt.legend()
    plt.grid(True, linestyle=':')

plt.suptitle('Central Limit Theorem (CLT) Verification (Uniform Distribution)', fontsize=14, y=1.05)
plt.tight_layout()
plt.savefig('clt_verification.png', bbox_inches='tight')
plt.close()
print("CLT visualization saved as 'clt_verification.png'")

=== 3. Central Limit Theorem Verification ===
CLT visualization saved as 'clt_verification.png'
